# 07. Real RAVE Inference

This notebook uses the actual `rave` CLI to run inference with real checkpoints.
It supports both:
- server mode (with existing `session_report.json` from NB02), and
- standalone mode (fallback input WAV via `PIANOKIT_RAVE_INPUT_WAV`).

Outputs:
- `artifacts/collab_rave/*.wav`
- `artifacts/collab_rave/style_metadata.json`
- `artifacts/collab_rave/stage_extension.json`


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import shlex
import sys

import numpy as np
import soundfile as sf
import librosa
import IPython.display as ipd

IS_COLAB = 'google.colab' in sys.modules
RUN_MODE = os.environ.get('PIANOKIT_RUN_MODE', 'colab' if IS_COLAB else 'server')
REPO_URL = os.environ.get('PIANOKIT_REPO_URL', 'https://github.com/jhbae/pianokit.git')

if IS_COLAB and RUN_MODE == 'colab' and not Path('assets').exists():
    repo_dir = Path('/content/pianokit')
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
    os.chdir(str(repo_dir))
    print(f'[bootstrap] changed cwd to {repo_dir}')

if IS_COLAB and RUN_MODE == 'colab':
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'acids-rave', 'librosa', 'soundfile', 'ipython'
    ], check=False)

ARTIFACTS = Path(os.environ.get('PIANOKIT_ARTIFACTS_DIR', 'artifacts'))
ASSETS = Path(os.environ.get('PIANOKIT_ASSETS_DIR', 'assets'))
SOMAX_DIR = ARTIFACTS / 'collab_somax'
RAVE_DIR = ARTIFACTS / 'collab_rave'
RAVE_DIR.mkdir(parents=True, exist_ok=True)

session_report_path = SOMAX_DIR / 'session_report.json'
if session_report_path.exists():
    session_report = json.loads(session_report_path.read_text())
    input_wav = Path(session_report.get('output_wav', ''))
    if not input_wav.exists():
        raise FileNotFoundError(
            f'Missing Somax WAV output: {input_wav}. '
            'Run NB02 or set PIANOKIT_RAVE_INPUT_WAV.'
        )
else:
    # Standalone fallback for Colab/server quick tests.
    fallback_wav = Path(os.environ.get('PIANOKIT_RAVE_INPUT_WAV', ASSETS / 'satie_gymnopedie_no1.wav'))
    if not fallback_wav.exists():
        raise FileNotFoundError(
            'No session_report.json and fallback WAV not found. '
            'Run NB02 first, or set PIANOKIT_RAVE_INPUT_WAV to a valid WAV path.'
        )
    input_wav = fallback_wav
    session_report = {
        'session_id': 'standalone_rave',
        'piece': 'satie',
        'phrase_boundaries': [],
    }

# IMPORTANT:
# 1) Provide real checkpoint paths for each profile.
# 2) Optionally set PIANOKIT_RAVE_MODELS_JSON to a JSON dict.
model_env = os.environ.get('PIANOKIT_RAVE_MODELS_JSON', '').strip()
if model_env:
    RAVE_MODEL_PATHS = json.loads(model_env)
else:
    RAVE_MODEL_PATHS = {
        'warm_legato': '',
        'percussive_edge': '',
        'ambient_tail': '',
    }

print(f'Run mode: {RUN_MODE} (colab={IS_COLAB})')
print('Input WAV:', input_wav)
print('RAVE profiles:', list(RAVE_MODEL_PATHS.keys()))


In [ ]:
missing = [k for k, v in RAVE_MODEL_PATHS.items() if not str(v).strip()]
if missing:
    raise ValueError(
        'Set real RAVE model paths before execution. Missing: ' + ', '.join(missing) + '\n'
        'Tip: set PIANOKIT_RAVE_MODELS_JSON env var with a JSON dict.'
    )

for k, v in RAVE_MODEL_PATHS.items():
    p = Path(str(v)).expanduser()
    if not p.exists():
        raise FileNotFoundError(
            f'Model path not found for {k}: {p}. '
            'Check checkpoint path on server/colab runtime.'
        )
    RAVE_MODEL_PATHS[k] = str(p)

print('All model paths validated.')


In [ ]:
rendered = {}
for profile, model_path in RAVE_MODEL_PATHS.items():
    profile_out = RAVE_DIR / profile
    profile_out.mkdir(parents=True, exist_ok=True)

    cmd = [
        'rave', 'generate',
        f'--model={model_path}',
        f'--input={str(input_wav)}',
        f'--out_path={str(profile_out)}',
        f'--name={profile}',
        '--gpu=-1',
    ]
    print('Running:', ' '.join(shlex.quote(c) for c in cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stdout)
        print(proc.stderr)
        raise RuntimeError(f'RAVE generate failed for profile {profile}')

    candidates = sorted(profile_out.rglob(input_wav.name))
    if not candidates:
        alt = sorted(profile_out.rglob('*.wav'))
        if not alt:
            raise FileNotFoundError(f'No generated wav found for profile {profile} in {profile_out}')
        out_wav = alt[0]
    else:
        out_wav = candidates[0]

    piece_name = session_report.get('piece', 'unknown')
    final_wav = RAVE_DIR / f'{piece_name}_{profile}.wav'
    data, sr = sf.read(out_wav)
    sf.write(final_wav, data, sr)
    rendered[profile] = str(final_wav)
    print('Saved profile wav:', final_wav)


In [ ]:
def audio_stats(path: str):
    y, sr = librosa.load(path, sr=44100, mono=True)
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    rms = librosa.feature.rms(y=y)[0]
    return {
        "duration_sec": float(len(y)/sr),
        "centroid_mean": float(np.mean(centroid)),
        "rms_mean": float(np.mean(rms)),
    }

style_meta = {
    "session_id": session_report["session_id"],
    "piece": session_report["piece"],
    "input_source": str(input_wav),
    "render_version": "rave-real-v1",
    "styles": {},
}

for profile, wav_path in rendered.items():
    style_meta["styles"][profile] = {
        "file": wav_path,
        "stats": audio_stats(wav_path),
        "model_path": RAVE_MODEL_PATHS[profile],
    }

meta_path = RAVE_DIR / "style_metadata.json"
meta_path.write_text(json.dumps(style_meta, ensure_ascii=False, indent=2))
print("Saved metadata:", meta_path)


In [ ]:
contract_path = ARTIFACTS / 'collab_contract.json'
if contract_path.exists():
    contract_version = json.loads(contract_path.read_text()).get('contract_version', 'unknown')
else:
    contract_version = 'unknown'

stage_extension = {
    'session_id': session_report.get('session_id', 'standalone_rave'),
    'piece': session_report.get('piece', 'unknown'),
    'phrase_boundaries': session_report.get('phrase_boundaries', []),
    'style_profile_candidates': list(rendered.keys()),
    'style_metadata_file': str(RAVE_DIR / 'style_metadata.json'),
    'consumes_contract_version': contract_version,
    'nb05_merge_hint': 'Merge into performance_timeline.json with key collab_stage_extension',
}

stage_path = RAVE_DIR / 'stage_extension.json'
stage_path.write_text(json.dumps(stage_extension, ensure_ascii=False, indent=2))
print('Saved stage extension:', stage_path)

print('Input preview')
display(ipd.Audio(filename=str(input_wav)))
for profile, wav_path in rendered.items():
    print(profile)
    display(ipd.Audio(filename=wav_path))
